In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_peak_search
from fooof import FOOOFGroup

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

fit foof for each subject for each channel separately but across trials.
In the end you arrive at an exponent and offset value for each trial per channel per subject.
THen correlate the offset and the exponent with the predicted amplitude?

In [ ]:
def extract_periodic(data, min_peak_height=0.05, max_n_peaks=6, peak_width_limits=[2.2, 8],aperiodic_mode='fixed'):
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, 1000, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False, bandwidth=10)
    periodic_components = []
    freqs = []
    errors = []
    r2s = []
    for ch_idx in range(data.shape[1]):
        fg = FOOOFGroup(peak_width_limits=peak_width_limits, min_peak_height=min_peak_height, max_n_peaks=max_n_peaks, aperiodic_mode=aperiodic_mode)
        fg.fit(freqs, psd[:,ch_idx], [2, 45], n_jobs=-1)
        for t in range(data.shape[0]):
            fm = fg.get_fooof(ind=t, regenerate=True)
            periodic_components.append(fm._peak_fit)


        errors_mean_channel = np.mean(fg.get_params("error"))
        r2s_mean_channel = np.mean(fg.get_params("r_squared"))
        errors.append(errors_mean_channel)
        r2s.append(r2s_mean_channel)

    return periodic_components, errors, r2s

In [ ]:
def extract_periodic2(data, min_peak_height=0.05, max_n_peaks=6, peak_width_limits=[2.2, 8],aperiodic_mode='fixed'):
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, 1000, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False, bandwidth=10)
    freqs = np.array(freqs)
    periodic_components = np.zeros((data.shape[0], data.shape[1], len(freqs)))
    errors = np.zeros((data.shape[0], data.shape[1]))
    r2s = np.zeros((data.shape[0], data.shape[1]))
    for ch_idx in range(data.shape[1]):
        for t in range(data.shape[0]):
            fm = FOOOF(peak_width_limits=peak_width_limits, min_peak_height=min_peak_height, max_n_peaks=max_n_peaks, aperiodic_mode=aperiodic_mode)
            fm.fit(freqs, psd[t,ch_idx], [2, 45])
            periodic_components[t,ch_idx] = fm._peak_fit
            errors[t,ch_idx] = fm.error_
            r2s[t,ch_idx] = fm.r_squared_

    errors_means= np.mean(errors, axis=0)
    r2s_means = np.mean(r2s, axis=0)

    return freqs, periodic_components, errors_means, r2s_means

In [ ]:
def total_bandpower_periodic(freqs, data, band, window_sec=None, relative=False):

    band = np.asarray(band)
    low, high = band

    freq_res = np.round(42/len(freqs),1)
    idx_band = np.logical_and(freqs >= low, freqs <= high)

    integral_band = simpson(data[idx_band], dx=freq_res)
    average_band = np.mean(data[idx_band])

    return integral_band, average_band

# subject 2 analysis

In [ ]:
save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data"
file_path = os.path.join(save_path, "subject_002_preprocessed_combined_py.fif")
#data = mne.read_epochs(file_path)
#epochs = data.get_data()[150:,:,:900]
cfg = load_config()
cfg.dataset.data_directory = save_path
cfg.dataset.subject_index = 2
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:,:,:900]

In [ ]:
def load_data(subject_index, save_path="/home/marco/Documents/GitHub/tms_eeg_decoding/data"):
    file_path = os.path.join(save_path, "subject_002_preprocessed_combined_py.fif")
#data = mne.read_epochs(file_path)
#epochs = data.get_data()[150:,:,:900]
    cfg = load_config()
    cfg.dataset.subject_index = subject_index

    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]
    return all_epochs, ch_names

In [ ]:
def process_subject_data(freqs, periodic_component,freq_bands):
    all_freq_bannds_integral = {freq: np.zeros((periodic_component.shape[0], periodic_component.shape[1])) for freq in freq_bands.keys()}
    all_freq_bannds_average = {freq: np.zeros((periodic_component.shape[0], periodic_component.shape[1])) for freq in freq_bands.keys()}
    for trial in range(all_epochs.shape[0]):
        for idx, ch in enumerate(ch_names):
            data = periodic_component[trial, idx, :]
            for band_name, band_range in freq_bands.items():
                integral, average = total_bandpower_periodic(freqs, data, band_range)
                all_freq_bannds_integral[band_name][trial, idx] = integral
                all_freq_bannds_average[band_name][trial, idx] = average
        
    return all_freq_bannds_integral, all_freq_bannds_average

In [ ]:
freq_bands = {"delta" : (2,4),
                "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

#all_epochs_2, ch_nams_2 = load_data(2)
#freqs, periodic_component, errors, r2s = extract_periodic2(all_epochs_2)
#all_freq_bannds_integral, all_freq_bannds_average = process_subject_data(freqs, periodic_component, freq_bands)


freq_bands = {"delta": (0.5,4)}

cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    all_epochs, ch_names = load_data(subject_index)
    freqs, periodic_component, errors, r2s = extract_periodic2(all_epochs)
    all_freq_bannds_integral, all_freq_bannds_average = process_subject_data(freqs, periodic_component, freq_bands)
    
    np.save(f"/home/marco/Documents/GitHub/tms_eeg_decoding/aperiodic_components analysis/delta_integral_{subject_index}.npy", all_freq_bannds_integral)
    np.save(f"/home/marco/Documents/GitHub/tms_eeg_decoding/aperiodic components analysis/delta_average_{subject_index}.npy", all_freq_bannds_average)

In [ ]:
#np.save(f"/home/marco/Documents/GitHub/tms_eeg_decoding/aperiodic components analysis/integral_{subject_index}.npy", all_freq_bannds_integral)
#np.save(f"/home/marco/Documents/GitHub/tms_eeg_decoding/aperiodic  components analysis/average_{subject_index}.npy", all_freq_bannds_average)

In [ ]:
import pickle
all_subject_amplitude_data = {}
all_subject_uncertainty_data = {}
cfg = load_config()
data_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject in cfg.dataset.test_subject_indices:
    with open(os.path.join(data_path, f"subject_{subject}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        all_subject_amplitude_data[subject] = data['predictions']
        all_subject_uncertainty_data[subject] = data['uncertainties']


In [ ]:
#cwd = os.getcwd()
#all_freq_bands_integral= np.load(os.path.join(cwd, "periodic_power", "integral_2.npy" ), allow_pickle=True).item()

In [ ]:
from scipy.stats import pearsonr
def plot_power_amplitude(subject_power_data, all_subject_amplitude_data,subject_index=2, show_plot=True, corr_treshold=0.3):
    all_amplitudes = all_subject_amplitude_data[subject_index]

    # data is of the shape subject_index, trial, band, channel
    # make separate plot for each channel and separate subplot for each frequency band where power is on the x-axis and amplitude is on the y-axis
    _, ch_names = load_data(subject_index)

    freq_bands = {"delta": (2, 4),
        "theta": (4, 8),
                    "alpha": (8, 12),
                    "beta": (12, 30),
                    "gamma": (30, 45)}

    high_correlations = {f: {} for f in freq_bands.keys()}
    corrs = {f: {} for f in freq_bands.keys()}
    corrs_abs = {f: {} for f in freq_bands.keys()}
    for band_idx, band_name in enumerate(freq_bands.keys()):
        corrs[band_name] = {}
        corrs_abs[band_name] = {}
        for ch_idx,ch in enumerate(ch_names):
            corrs[band_name][ch] = {}
            corrs_abs[band_name][ch] = {}
            if show_plot:
                fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(10, 2), sharey=True)
                fig.suptitle(f"Channel: {ch}")
        
            

            current_data = subject_power_data[band_name][:, ch_idx]
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr, pval = pearsonr(current_data[indices], all_amplitudes[indices])
            if np.abs(corr) >= corr_treshold:
                high_correlations[band_name][ch] = corr
            corrs_abs[band_name][ch]["stat"] = np.abs(corr)
            corrs[band_name][ch]["stat"] = corr
            corrs_abs[band_name][ch]["pval"] = pval
            corrs[band_name][ch]["pval"] = pval

            if show_plot:
                axs[band_idx].set_xlabel('Power')
                axs[band_idx].set_ylabel('Amplitude')
                if np.abs(corr) >= corr_treshold:
                    axs[band_idx].set_title(f"{band_name}, corr: {corr:.2f}")
                    
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')

                else:
                    axs[band_idx].set_title(f"{band_name}")
                    axs[band_idx].scatter(current_data[indices], all_amplitudes[indices], alpha=0.5)
            #fig.savefig(f"correlation_plots/channel_{ch}_subject_{subject_index}.png")
            plt.show()
    return high_correlations, corrs, corrs_abs
        

In [ ]:
def plot_power_amplitude_filtered(subject_power_data, all_subject_amplitude_data, subject_index=2, corr_threshold=0.3, show_plot=True):
    all_amplitudes = all_subject_amplitude_data[subject_index]
    _, ch_names = load_data(subject_index)
    freq_bands = {"theta": (4, 8),
                  "alpha": (8, 12),
                  "beta": (12, 30),
                  "gamma": (30, 45)}

    high_correlations = {f: {} for f in freq_bands.keys()}

    for ch_idx, ch in enumerate(ch_names):
        for band_idx, band_name in enumerate(freq_bands.keys()):
            current_data = subject_power_data[band_name][:, ch_idx]
            all_amplitudes = np.array(all_amplitudes)
            top_1_percent = np.percentile(current_data, 99)
            indices = current_data <= top_1_percent
            corr = np.corrcoef(current_data[indices], all_amplitudes[indices])[0, 1]

            if corr_threshold == -1 or np.abs(corr) >= corr_threshold:
                high_correlations[band_name][ch] = corr
                if show_plot:
                    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 3))
                    fig.suptitle(f"Subject {subject_index}, Channel: {ch}")
                    ax.set_xlabel('Power')
                    ax.set_ylabel('Amplitude')
                    ax.set_title(f"{band_name}, corr: {corr:.2f}", y=0.95)
                    ax.scatter(current_data[indices], all_amplitudes[indices], alpha=0.5, c='k')
                    os.makedirs("correlation_plots", exist_ok=True)
                    # fig.savefig(f"correlation_plots/subject_{subject_index}_channel_{ch}_band_{band_name}.png")

    return high_correlations


In [ ]:
#plot_power_amplitude(all_freq_bands_integral, all_subject_amplitude_data, subject_index=2)

In [ ]:
#plot_power_amplitude_filtered(all_freq_bands_integral, all_subject_amplitude_data, subject_index=2)

# all subjects

In [ ]:
cwd = os.getcwd()

all_subjects_highest_corrs_all = {}
all_subjects_highest_corrs_all_abs = {}
all_subjects_sig_channels = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    all_freq_bands_integral= np.load(os.path.join("/home/marco/Documents/GitHub/tms_eeg_decoding/aperiodic components analysis", f"integral_{subject_index}.npy" ), allow_pickle=True).item()
    delta_integral = np.load(os.path.join(cwd, f"delta_integral_{subject_index}.npy" ), allow_pickle=True).item()
    merged_freq_bands = {**all_freq_bands_integral, **delta_integral}
    #all_subjects_sig_channels[subject_index], all_subjects_highest_corrs_all[subject_index], all_subjects_highest_corrs_all_abs[subject_index]  = #plot_power_amplitude(all_freq_bands_integral, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)
    all_subjects_sig_channels[subject_index], all_subjects_highest_corrs_all[subject_index], all_subjects_highest_corrs_all_abs[subject_index] = plot_power_amplitude(merged_freq_bands, all_subject_amplitude_data, subject_index=subject_index, show_plot=False)
    

In [ ]:
np.save('all_subjects_highest_corrs_abs_perodic.npy', all_subjects_highest_corrs_all_abs)
np.save('all_subjects_highests_corrs_periodic.npy', all_subjects_highest_corrs_all)

In [ ]:
all_subjects_highest_corrs_all

In [ ]:
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_periodic.npy"), all_subjects_highest_corrs_all)
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_abs_periodic.npy"), all_subjects_highest_corrs_all_abs)

In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}



In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:
cfg = load_config()
top_10_per_subject = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject.append(top_10_channels)

In [ ]:
def extract_top_10_channels_and_mean_corr(all_subjects_corrs, k=60):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, freq_bands in all_subjects_corrs.items():
        top_10_channels_per_subject[subject] = {}
        mean_corr_per_subject[subject] = {}
        
        for band, channels in freq_bands.items():
            sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:k]
            top_10_channels_per_subject[subject][band] = dict(sorted_channels)
            
            mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
            mean_corr_per_subject[subject][band] = mean_corr
    
    return top_10_channels_per_subject, mean_corr_per_subject

top_10_channels_per_subject_abs, mean_corr_per_subject_abs = extract_top_10_channels_and_mean_corr(all_subjects_highest_corrs_all_abs)

In [ ]:
top_10_channels_per_subject, mean_corr_per_subject = extract_top_10_channels_and_mean_corr(all_subjects_highest_corrs_all)

In [ ]:
def compute_summary_top_10(top_10, k=60):
    # across subjects and channels compute the min value, max value, the mean of the absolute values and the standard deviation
    all_corrs = {band:[] for band in top_10[1].keys()}
    all_corrs_abs = {band:[] for band in top_10[1].keys()}
    for subject, bands in top_10.items():
        for band, channels in bands.items():
            for channel, value in channels.items():
                all_corrs[band].append(value)
                all_corrs_abs[band].append(abs(value))
    
    summary = {}
    for band in all_corrs.keys():
        summary[band] = {}
        summary[band]["min"] = np.nanmin(all_corrs[band])
        summary[band]["max"] = np.nanmax(all_corrs[band])
        summary[band]["mean"] = np.nanmean(all_corrs_abs[band])
        summary[band]["std"] = np.nanstd(all_corrs_abs[band])
    return summary
                
                

In [ ]:
compute_summary_top_10(top_10_channels_per_subject)

In [ ]:
np.save("top_10_periodic_component",top_10_channels_per_subject)

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, frequency_band='gamma'):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print(f"Channels with high correlation in {frequency_band} band:", list(all_subjects_highest_corrs[subject_index][frequency_band].keys()))

        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index][frequency_band].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index][frequency_band].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"Number of common channels: {num_common_channels}")
        print(f"Ratio: {ratio:.2f}")
        print()
    
    empty_list_indices = subject_common_channels[subject_common_channels ==0]
    fig = plt.figure(figsize=(10, 5))
    plt.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            plt.plot(idx, subject_ratios[idx], 'ro')

    plt.xticks(np.arange(len(subject_ratios)), cfg.dataset.test_subject_indices)
    plt.xlabel('Subject Index')
    plt.xticks(rotation=45)
    plt.ylabel('Ratio of Common Channels')
    plt.title(f'Common Channels Ratio for {frequency_band} Band')
    plt.savefig(f"common_channels_ratio_{frequency_band}_periodic_component.png")
    plt.show()

# Example usage:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='gamma')



In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='beta')


In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='alpha')


In [ ]:
compare_channels(cfg, top_10_per_subject, top_10_channels_per_subject, frequency_band='theta')

# rank correlations of all 60 channels

In [ ]:
cfg = load_config()
top_60_per_subject = []
top_60_per_subject_dict = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_60_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 60, update_channel_points_linear)
    top_60_per_subject.append(top_60_channels)
    top_60_per_subject_dict.append(sum_over_channel_points_dict)

In [ ]:
np.save(all_subjects_highest_corrs_all)

In [ ]:
top_60_per_subject_dict = {subject: top_60_per_subject_dict[i] for i, subject in enumerate(cfg.dataset.test_subject_indices)}

In [ ]:
from scipy.stats import spearmanr
rank_correlations = {}

for freq_band in freq_bands.keys():
    rank_correlations[freq_band] = {}
    for subject_index in cfg.dataset.test_subject_indices:
        top_60_channels = top_60_per_subject_dict[subject_index]
        corrs = all_subjects_highest_corrs_all_abs[subject_index][freq_band]
        print(subject_index)

        # Compute the Spearman rank correlation
        
        rank_corr, pval = spearmanr(list(top_60_channels.values()), list(corrs.values()))
        print(f"Rank correlation: {rank_corr:.2f}, p-value: {pval:.2f}")
      
        
        rank_correlations[freq_band][subject_index] = (rank_corr, pval)

In [ ]:
import matplotlib.pyplot as plt


# Plot rank correlations for each frequency band
for freq_band in freq_bands:
    plt.figure(figsize=(12, 6))
    subjects = list(rank_correlations[freq_band].keys())
    rank_corrs = [rank_correlations[freq_band][subject][0] for subject in subjects]
    p_values = [rank_correlations[freq_band][subject][1] for subject in subjects]

    plt.bar(subjects, rank_corrs, color='blue', alpha=0.7, label='Rank Correlation')
    plt.scatter(subjects, rank_corrs, c=['red' if p < (0.05/60) else 'black' for p in p_values], label='p < 0.05', zorder=5)
    
    plt.xlabel('Subject Index')
    plt.ylabel('Rank Correlation')
    plt.title(f'Rank Correlation for {freq_band} Band')
    plt.axhline(y=0, color='gray', linestyle='--')
    plt.legend()
    plt.show()